# Fase 3 -- Feature engineering temporal

**Notebook**: `05_feature_engineering.ipynb`  
**Asignatura**: Inteligencia Artificial -- Universidad de Ibague

Construye `data/processed/colombia_features_ml.csv` desde:

1. Indicadores financieros (Fase 1).
2. Etiquetas de riesgo (Fase 2).
3. Metadata empresarial del consolidado SIREM (CIIU, departamento, tipo societario).

Salida (203,104 filas x 77 columnas):

- 3 IDs (NIT_LIMPIO, ANIO, etiqueta_final)
- 18 indicadores base + Z''-Score Altman
- **18 deltas interanuales** (variacion relativa anyo a anyo)
- **6 tasas de crecimiento** (2 y 3 anyos para roa, margen_neto, razon_deuda)
- 4 features de **escala/estructura** (log-tamano + estructura de balance)
- 2 features temporales (anyo numerico + dummy COVID 2020)
- 22 dummies categoricas (CIIU top10 + 'otros' + 'sin_ciiu' = 12, sociedad x4, depto top5 + otros = 6)
- 3 columnas de texto categoricas (ciiu_seccion, sociedad, departamento) que sirven para Fase 8 (estabilidad sectorial), redundantes para el modelo.

**Total features utilizables por ML (sin IDs ni texto): 71.**

Criterios de aceptacion (plan §3.3):

1. No hay columnas con 100% nulos.
2. No hay duplicados (NIT_LIMPIO, ANIO).
3. Las dummies suman 1 por fila dentro de cada grupo categorico.

In [1]:
from __future__ import annotations
import json
import os
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from src.etl_utils import cargar_consolidado

np.random.seed(42)
pd.set_option('display.max_columns', 80)

# --- listas de variables base ---
INDICADORES_BASE = [
    'razon_corriente', 'prueba_acida', 'capital_trabajo', 'razon_efectivo',
    'margen_bruto', 'margen_operacional', 'margen_neto', 'roa', 'roe',
    'razon_deuda', 'deuda_patrimonio', 'apalancamiento', 'cobertura_intereses',
    'rotacion_activos', 'rotacion_inventarios', 'rotacion_cartera',
    'dias_inventario', 'dias_cartera',
]
INDICADORES_TENDENCIA = ['roa', 'margen_neto', 'razon_deuda']

print('ROOT:', ROOT)

ROOT: /home/Retrius/Documents/Personal Projects/trabajo final ia


## 1. Carga de fuentes

- `colombia_indicadores_pymes.csv` (Fase 1): 18 ratios + Z'' por (NIT, anyo).
- `colombia_etiquetas_riesgo.csv`   (Fase 2): `etiqueta_final` por (NIT, anyo).
- `colombia_consolidado_pymes.csv`  (ETL): 5 columnas financieras de balance + 3 de metadata.

Las columnas de metadata del consolidado tienen mojibake distinto al de los
estados financieros (caracter U+FFFD `ï¿½` en lugar de vocales tildadas), por
lo que se localizan por substring.

In [2]:
indicadores = pd.read_csv(
    ROOT / 'data' / 'processed' / 'colombia_indicadores_pymes.csv',
    low_memory=False,
)
etiquetas = pd.read_csv(
    ROOT / 'data' / 'processed' / 'colombia_etiquetas_riesgo.csv',
    low_memory=False,
    usecols=['NIT_LIMPIO', 'ANIO', 'etiqueta_final'],
)

cols_consolidado = [
    'NIT_LIMPIO', 'ANIO',
    'Total de activos', 'Activos corrientes totales', 'Total pasivos',
    'Pasivos corrientes totales', 'Ingresos de actividades ordinarias',
]
df_head = pd.read_csv(
    ROOT / 'data' / 'processed' / 'colombia_consolidado_pymes.csv',
    low_memory=False, nrows=0,
)
md_aliases = {}
for clave, patron in [
    ('CIIU',         'Industrial Internacional'),
    ('DEPARTAMENTO', 'Departamento de la direcci'),
    ('SOCIEDAD',     'Tipo societario'),
]:
    candidatos = [c for c in df_head.columns if patron in c]
    if clave == 'DEPARTAMENTO':
        candidatos = [c for c in candidatos if 'domicilio' in c]
    md_aliases[clave] = candidatos[0]
cols_consolidado.extend(md_aliases.values())

consolidado = cargar_consolidado(
    path=str(ROOT / 'data' / 'processed' / 'colombia_consolidado_pymes.csv'),
    usecols=cols_consolidado,
)
consolidado = consolidado.rename(columns={
    'Total de activos':                   'total_activos',
    'Activos corrientes totales':         'activos_corrientes',
    'Total pasivos':                      'total_pasivos',
    'Pasivos corrientes totales':         'pasivos_corrientes',
    'Ingresos de actividades ordinarias': 'ingresos',
    md_aliases['CIIU']:                   'ciiu_raw',
    md_aliases['DEPARTAMENTO']:           'departamento_raw',
    md_aliases['SOCIEDAD']:               'sociedad_raw',
})

print(f'indicadores: {indicadores.shape}')
print(f'etiquetas:   {etiquetas.shape}')
print(f'consolidado: {consolidado.shape}')
consolidado.head(3)

indicadores: (203104, 23)
etiquetas:   (203104, 3)
consolidado: (203104, 10)


,NIT_LIMPIO,ANIO,ciiu_raw,departamento_raw,sociedad_raw,activos_corrientes,pasivos_corrientes,total_activos,total_pasivos,ingresos
0,800000090,2016,F4111 - Construcci�n de edificios residenciales,ANTIOQUIA,08. SOCIEDAD POR ACCIONES SIMPLIFICADA SAS,5355290.0,1756752.0,6542076.0,5352469.0,1562353.0
1,800000090,2018,F4111 - Construcci�n de edificios residenciales,ANTIOQUIA,08. SOCIEDAD POR ACCIONES SIMPLIFICADA SAS,6359301.0,1623320.0,7341985.0,6007115.0,1674930.0
2,800000090,2019,F4111 - Construcci�n de edificios residenciales,ANTIOQUIA,08. SOCIEDAD POR ACCIONES SIMPLIFICADA SAS,8191026.0,2394292.0,9000225.0,7570034.0,2319484.0


## 2. Variaciones interanuales (delta)

Para cada uno de los 18 indicadores: `(x_t - x_{t-1}) / |x_{t-1}|`.  
El primer anyo por empresa queda como `NaN` (no hay anterior).  
Cuando `x_{t-1} == 0` se devuelve `NaN` (evita `inf`).  
`+inf`/`-inf` residuales se convierten a `NaN`.

In [3]:
def variaciones_interanuales(df, indicadores):
    df = df.sort_values(['NIT_LIMPIO', 'ANIO']).copy()
    g = df.groupby('NIT_LIMPIO', sort=False)
    out = pd.DataFrame(index=df.index)
    for col in indicadores:
        prev = g[col].shift(1)
        denom = prev.abs().where(prev != 0)
        out[f'{col}_d1'] = (df[col] - prev) / denom
    return out.replace([np.inf, -np.inf], np.nan)

deltas = variaciones_interanuales(indicadores, INDICADORES_BASE)
print('deltas shape:', deltas.shape)
deltas.head(3)

deltas shape: (203104, 18)


,razon_corriente_d1,prueba_acida_d1,capital_trabajo_d1,razon_efectivo_d1,margen_bruto_d1,margen_operacional_d1,margen_neto_d1,roa_d1,roe_d1,razon_deuda_d1,deuda_patrimonio_d1,apalancamiento_d1,cobertura_intereses_d1,rotacion_activos_d1,rotacion_inventarios_d1,rotacion_cartera_d1,dias_inventario_d1,dias_cartera_d1
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0.285088,0.200441,0.316085,2.518509,0.071426,-0.323929,3.836668,3.620254,3.620919,0.000032,0.000176,0.000144,0.651277,-0.044744,-0.968774,-0.027341,31.024834,0.028110
2,-0.126715,-0.217374,0.223977,-0.293496,-0.221229,0.003609,0.064641,0.202702,0.376077,0.027997,0.176188,0.144155,-0.103664,0.129678,48.078442,0.198550,-0.979624,-0.165658


## 3. Tasas de crecimiento multi-anyo (g2, g3)

Para los indicadores clave de rentabilidad y endeudamiento (`roa`,
`margen_neto`, `razon_deuda`), capturar tendencia a 2 y 3 anyos:

$$\text{growth}_k = \frac{x_t - x_{t-k}}{|x_{t-k}|}$$

In [4]:
def crecimiento_multi_anyo(df, indicadores):
    df = df.sort_values(['NIT_LIMPIO', 'ANIO']).copy()
    g = df.groupby('NIT_LIMPIO', sort=False)
    out = pd.DataFrame(index=df.index)
    for k in (2, 3):
        for col in indicadores:
            prev = g[col].shift(k)
            denom = prev.abs().where(prev != 0)
            out[f'{col}_g{k}'] = (df[col] - prev) / denom
    return out.replace([np.inf, -np.inf], np.nan)

crecimientos = crecimiento_multi_anyo(indicadores, INDICADORES_TENDENCIA)
print('crecimientos shape:', crecimientos.shape)
crecimientos.head(3)

crecimientos shape: (203104, 6)


,roa_g2,margen_neto_g2,razon_deuda_g2,roa_g3,margen_neto_g3,razon_deuda_g3
0,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN
2,4.556787,4.149313,0.02803,NaN,NaN,NaN


## 4. Escala y estructura de balance

- `log_total_activos = log(1 + total_activos)`, idem para ingresos.
  Captura efectos de tamano sin la escala bruta en pesos.
- Estructura corriente del balance:
  - `ratio_activos_corr = activos_corrientes / total_activos`
  - `ratio_pasivos_corr = pasivos_corrientes / total_pasivos`

In [5]:
def features_escala_estructura(consolidado):
    df = consolidado[['NIT_LIMPIO', 'ANIO']].copy()
    df['log_total_activos'] = np.log1p(consolidado['total_activos'].clip(lower=0))
    df['log_ingresos']      = np.log1p(consolidado['ingresos'].clip(lower=0))
    den_act = consolidado['total_activos'].where(consolidado['total_activos'] != 0)
    den_pas = consolidado['total_pasivos'].where(consolidado['total_pasivos'] != 0)
    df['ratio_activos_corr'] = consolidado['activos_corrientes'] / den_act
    df['ratio_pasivos_corr'] = consolidado['pasivos_corrientes'] / den_pas
    return df.replace([np.inf, -np.inf], np.nan)

escala = features_escala_estructura(consolidado)
print('escala/estructura shape:', escala.shape)
escala.head(3)

escala/estructura shape: (203104, 6)


,NIT_LIMPIO,ANIO,log_total_activos,log_ingresos,ratio_activos_corr,ratio_pasivos_corr
0,800000090,2016,15.693765,14.261704,0.818592,0.328213
1,800000090,2018,15.809120,14.331283,0.866156,0.270233
2,800000090,2019,16.012760,14.656856,0.910091,0.316286


## 5. Dummies categoricas

- **CIIU**: extraer la letra de seccion (G, L, F, C, M, A, K, N, J, I) y
  agrupar el resto en `otros`. NaN -> `sin_ciiu`. **12 niveles**.
- **Tipo societario**: SAS (dominante, ~75% del universo), SA, Ltda, otra.
  **4 niveles**.
- **Departamento**: Bogota, Antioquia, Valle, Atlantico, Cundinamarca
  + `otros`. **6 niveles**.

Los textos categoricos se conservan ademas como columnas de texto
(`ciiu_seccion`, `sociedad`, `departamento`) para diagnostico sectorial
en Fase 8; el modelo solo usa los dummies.

In [6]:
CIIU_TOP_LETRAS_ETIQUETAS = {
    'G': 'comercio',           # Comercio al por mayor y al por menor
    'L': 'inmobiliario',       # Actividades inmobiliarias
    'F': 'construccion',       # Construccion
    'C': 'industria',          # Industrias manufactureras
    'M': 'profesional',        # Profesional, cientifica, tecnica
    'A': 'agricola',           # Agricultura, ganaderia, pesca
    'K': 'financiero',         # Financieras y seguros
    'N': 'apoyo_emp',          # Servicios administrativos y de apoyo
    'J': 'info_com',           # Informacion y comunicaciones
    'I': 'alojamiento',        # Alojamiento y comidas
}

def normalizar_categoricas(consolidado):
    df = consolidado[['NIT_LIMPIO', 'ANIO']].copy()

    letra = consolidado['ciiu_raw'].astype(str).str.extract(r'^([A-Z])', expand=False)
    df['ciiu_seccion'] = letra.map(CIIU_TOP_LETRAS_ETIQUETAS)
    df.loc[letra.isna(), 'ciiu_seccion'] = 'sin_ciiu'
    df['ciiu_seccion'] = df['ciiu_seccion'].fillna('otros')

    s = consolidado['sociedad_raw'].astype(str).str.upper().fillna('')
    sociedad = pd.Series('otra', index=df.index)
    sociedad[s.str.contains('SIMPLIFICADA SAS')] = 'sas'
    sociedad[
        s.str.contains(r'\bSOCIEDAD AN\W+NIMA\b', regex=True)
        & ~s.str.contains('SIMPLIFICADA')
    ] = 'sa'
    sociedad[s.str.contains('SOCIEDAD LIMITADA')] = 'ltda'
    df['sociedad'] = sociedad

    depto = consolidado['departamento_raw'].astype(str).str.upper().str.strip()
    mapeo = {
        'BOGOTA D.C.':  'bogota',
        'ANTIOQUIA':    'antioquia',
        'VALLE':        'valle',
        'ATLANTICO':    'atlantico',
        'CUNDINAMARCA': 'cundinamarca',
    }
    df['departamento'] = depto.map(mapeo).fillna('otros')
    return df

cats = normalizar_categoricas(consolidado)
dummies = pd.get_dummies(
    cats[['ciiu_seccion', 'sociedad', 'departamento']],
    prefix=['ciiu', 'sociedad', 'depto'], prefix_sep='_', dtype='int8',
)
print('cats:   ', cats.shape)
print('dummies:', dummies.shape)
print()
print('Distribucion ciiu_seccion (%):')
print((cats['ciiu_seccion'].value_counts(normalize=True) * 100).round(1))
print()
print('Distribucion sociedad (%):')
print((cats['sociedad'].value_counts(normalize=True) * 100).round(1))
print()
print('Distribucion departamento (%):')
print((cats['departamento'].value_counts(normalize=True) * 100).round(1))

cats:    (203104, 5)
dummies: (203104, 22)

Distribucion ciiu_seccion (%):
ciiu_seccion
comercio        23.2
inmobiliario    16.1
industria       12.3
construccion    12.2
profesional      7.4
agricola         6.0
sin_ciiu         5.2
otros            4.9
financiero       4.4
apoyo_emp        4.0
info_com         2.5
alojamiento      1.8
Name: proportion, dtype: float64

Distribucion sociedad (%):
sociedad
sas     75.1
sa      12.2
ltda     7.2
otra     5.5
Name: proportion, dtype: float64

Distribucion departamento (%):
departamento
bogota          42.9
otros           19.1
antioquia       17.3
valle            9.2
atlantico        6.1
cundinamarca     5.4
Name: proportion, dtype: float64


## 6. Merge final + winsorizacion

1. `indicadores` (sin `zona_altman_*`) + `deltas` + `crecimientos` por
   indice (mismo orden de filas).
2. + `escala` por `(NIT_LIMPIO, ANIO)`.
3. + columnas categoricas + dummies.
4. `anyo_num` (int16) + `covid_2020` (dummy).
5. `etiqueta_final` por `(NIT_LIMPIO, ANIO)`.
6. Winsorizacion p1-p99 sobre indicadores ratio, deltas, crecimientos,
   logs y ratios de estructura. **No** se winsoriza sobre dummies ni `anyo_num`.

In [7]:
def winsorizar(df, columnas, p_low=0.01, p_high=0.99):
    for col in columnas:
        if col not in df.columns:
            continue
        lo, hi = df[col].quantile([p_low, p_high])
        df[col] = df[col].clip(lower=lo, upper=hi)

base = indicadores.drop(
    columns=['zona_altman_original', 'zona_altman_terciles'], errors='ignore',
)
base = pd.concat([base, deltas, crecimientos], axis=1)
base = base.merge(escala, on=['NIT_LIMPIO', 'ANIO'], how='left')
base = base.merge(
    cats[['NIT_LIMPIO', 'ANIO', 'ciiu_seccion', 'sociedad', 'departamento']],
    on=['NIT_LIMPIO', 'ANIO'], how='left',
)
base = pd.concat([base.reset_index(drop=True),
                  dummies.reset_index(drop=True)], axis=1)
base['anyo_num']   = base['ANIO'].astype('int16')
base['covid_2020'] = (base['ANIO'] == 2020).astype('int8')
base = base.merge(etiquetas, on=['NIT_LIMPIO', 'ANIO'], how='left')

cols_winsorizar = (
    INDICADORES_BASE
    + ['z_score_altman']
    + [f'{c}_d1' for c in INDICADORES_BASE]
    + [f'{c}_g2' for c in INDICADORES_TENDENCIA]
    + [f'{c}_g3' for c in INDICADORES_TENDENCIA]
    + ['log_total_activos', 'log_ingresos',
       'ratio_activos_corr', 'ratio_pasivos_corr']
)
winsorizar(base, cols_winsorizar)

print('shape final:', base.shape)
base.head(3)

shape final: (203104, 77)


,NIT_LIMPIO,ANIO,razon_corriente,prueba_acida,capital_trabajo,razon_efectivo,margen_bruto,margen_operacional,margen_neto,roa,roe,razon_deuda,deuda_patrimonio,apalancamiento,cobertura_intereses,rotacion_activos,rotacion_inventarios,rotacion_cartera,dias_inventario,dias_cartera,z_score_altman,razon_corriente_d1,prueba_acida_d1,capital_trabajo_d1,razon_efectivo_d1,margen_bruto_d1,margen_operacional_d1,margen_neto_d1,roa_d1,roe_d1,razon_deuda_d1,deuda_patrimonio_d1,apalancamiento_d1,cobertura_intereses_d1,rotacion_activos_d1,rotacion_inventarios_d1,rotacion_cartera_d1,dias_inventario_d1,dias_cartera_d1,roa_g2,margen_neto_g2,razon_deuda_g2,roa_g3,margen_neto_g3,razon_deuda_g3,log_total_activos,log_ingresos,ratio_activos_corr,ratio_pasivos_corr,ciiu_seccion,sociedad,departamento,ciiu_agricola,ciiu_alojamiento,ciiu_apoyo_emp,ciiu_comercio,ciiu_construccion,ciiu_financiero,ciiu_industria,ciiu_info_com,ciiu_inmobiliario,ciiu_otros,ciiu_profesional,ciiu_sin_ciiu,sociedad_ltda,sociedad_otra,sociedad_sa,sociedad_sas,depto_antioquia,depto_atlantico,depto_bogota,depto_cundinamarca,depto_otros,depto_valle,anyo_num,covid_2020,etiqueta_final
0,800000090,2016,3.048404,2.632210,3598538.0,0.008661,0.930141,0.235361,0.007981,0.001906,0.010482,0.818161,4.499359,5.499359,1.350401,0.238816,0.149277,0.338984,2445.116085,1076.746180,4.251626,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15.693765,14.261704,0.818592,0.328213,construccion,sas,antioquia,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,2016,0,riesgo_medio
1,800000090,2018,3.917466,3.159811,4735981.0,0.030473,0.996577,0.159121,0.038601,0.008806,0.048435,0.818187,4.500150,5.500150,2.229886,0.228130,0.004661,0.329716,14412.820696,1107.013251,4.801962,0.285088,0.200441,0.316085,2.518509,0.071426,-0.323929,3.836668,3.620254,3.620919,0.000032,0.000176,0.000144,0.651277,-0.044744,-0.968774,-0.027341,31.024834,0.028110,NaN,NaN,NaN,NaN,NaN,NaN,15.809120,14.331283,0.866156,0.270233,construccion,sas,antioquia,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,2018,0,riesgo_medio
2,800000090,2019,3.421064,2.472952,5796734.0,0.021530,0.776106,0.159695,0.041096,0.010591,0.066650,0.841094,5.293023,6.293023,1.998727,0.257714,0.228769,0.395181,1595.495582,923.627108,4.851985,-0.126715,-0.217374,0.223977,-0.293496,-0.221229,0.003609,0.064641,0.202702,0.376077,0.027997,0.176188,0.144155,-0.103664,0.129678,33.136323,0.198550,-0.971141,-0.165658,4.556787,4.149313,0.02803,NaN,NaN,NaN,16.012760,14.656856,0.910091,0.316286,construccion,sas,antioquia,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,2019,0,riesgo_medio


## 7. Validaciones (criterios 3.3)

In [8]:
# 1. No columnas con 100% nulos.
nulos_pct = base.isna().mean() * 100
cols_100 = nulos_pct[nulos_pct == 100.0].index.tolist()
print(f'Columnas con 100% nulos: {len(cols_100)}  -> {cols_100}')
assert len(cols_100) == 0, 'Hay columnas 100% nulas'

# 2. Sin duplicados (NIT_LIMPIO, ANIO).
dup = int(base.duplicated(subset=['NIT_LIMPIO', 'ANIO']).sum())
print(f'Duplicados (NIT_LIMPIO, ANIO): {dup}')
assert dup == 0

# 3. Dummies suman 1 dentro de cada grupo categorico.
def _dummy_cols(prefix):
    return [c for c in base.columns
            if c.startswith(prefix) and base[c].dtype.kind == 'i']

for grupo in ('ciiu_', 'sociedad_', 'depto_'):
    cols = _dummy_cols(grupo)
    suma = base[cols].sum(axis=1)
    print(f'{grupo:<12}({len(cols):>2} dummies)  '
          f'rows con suma=1: {(suma == 1).sum():>7}, '
          f'suma=0: {(suma == 0).sum()}, '
          f'suma>=2: {(suma >= 2).sum()}')
    assert (suma == 1).all(), f'Dummies de {grupo} no suman 1 por fila'

print('\n[OK] Los 3 criterios de aceptacion 3.3 se cumplen.')

Columnas con 100% nulos: 0  -> []
Duplicados (NIT_LIMPIO, ANIO): 0
ciiu_       (12 dummies)  rows con suma=1:  203104, suma=0: 0, suma>=2: 0
sociedad_   ( 4 dummies)  rows con suma=1:  203104, suma=0: 0, suma>=2: 0
depto_      ( 6 dummies)  rows con suma=1:  203104, suma=0: 0, suma>=2: 0

[OK] Los 3 criterios de aceptacion 3.3 se cumplen.


## 8. Distribuciones de control

In [9]:
print('Etiqueta final por anyo:')
print(pd.crosstab(base['ANIO'], base['etiqueta_final'], margins=False))
print()
print('% etiqueta final (global):')
print((base['etiqueta_final'].value_counts(normalize=True) * 100).round(1))

Etiqueta final por anyo:
etiqueta_final  riesgo_alto  riesgo_bajo  riesgo_medio
ANIO                                                  
2016                   4253         2949         12501
2017                     56           15         16348
2018                   3538         2366         10043
2019                   4411         3533         13915
2020                   4976         4230         16788
2021                   4106         4889         16219
2022                   4419         5312         17194
2023                   4490         5385         17091
2024                   3928         5348         14801

% etiqueta final (global):
etiqueta_final
riesgo_medio    66.4
riesgo_alto     16.8
riesgo_bajo     16.8
Name: proportion, dtype: float64


In [10]:
nuevos = (
    [f'{c}_d1' for c in INDICADORES_BASE]
    + [f'{c}_g2' for c in INDICADORES_TENDENCIA]
    + [f'{c}_g3' for c in INDICADORES_TENDENCIA]
    + ['log_total_activos', 'log_ingresos',
       'ratio_activos_corr', 'ratio_pasivos_corr']
)
comp = (1 - base[nuevos].isna().mean()) * 100
comp = comp.sort_values()
print('Completitud de NUEVOS features (% no-nulos), peores 10:')
print(comp.head(10).round(1).to_string())
print('\nCompletitud de NUEVOS features, mejores 10:')
print(comp.tail(10).round(1).to_string())

Completitud de NUEVOS features (% no-nulos), peores 10:
dias_inventario_d1         42.9
rotacion_inventarios_d1    43.3
roa_g3                     43.5
razon_deuda_g3             43.6
margen_neto_g3             46.6
cobertura_intereses_d1     49.0
roa_g2                     57.4
razon_deuda_g2             57.6
margen_neto_g2             59.8
rotacion_activos_d1        62.3

Completitud de NUEVOS features, mejores 10:
apalancamiento_d1         66.4
dias_cartera_d1           71.7
rotacion_cartera_d1       72.9
margen_bruto_d1           74.8
margen_neto_d1            74.9
margen_operacional_d1     74.9
ratio_pasivos_corr        90.8
ratio_activos_corr        91.7
log_total_activos         91.7
log_ingresos             100.0


## 9. Guardado

In [11]:
out_path = ROOT / 'data' / 'processed' / 'colombia_features_ml.csv'
base.to_csv(out_path, index=False, encoding='utf-8')
print(f'Guardado: {out_path.relative_to(ROOT)}')
print(f'  shape:   {base.shape}')
print(f'  size:    {out_path.stat().st_size / 1024 / 1024:.1f} MB')

Guardado: data/processed/colombia_features_ml.csv
  shape:   (203104, 77)
  size:    147.7 MB


In [12]:
# Resumen JSON para reportes/informe.
os.makedirs(ROOT / 'reports' / 'metrics', exist_ok=True)
ids = ['NIT_LIMPIO', 'ANIO', 'etiqueta_final']
deltas_cols    = [c for c in base.columns if c.endswith('_d1')]
g2g3_cols      = [c for c in base.columns if c.endswith('_g2') or c.endswith('_g3')]
escala_cols    = ['log_total_activos', 'log_ingresos',
                  'ratio_activos_corr', 'ratio_pasivos_corr']
anyo_cols      = ['anyo_num', 'covid_2020']
cats_txt_cols  = ['ciiu_seccion', 'sociedad', 'departamento']
dummy_cols     = (_dummy_cols('ciiu_') + _dummy_cols('sociedad_')
                  + _dummy_cols('depto_'))
total_ml = (len(INDICADORES_BASE) + 1 + len(deltas_cols) + len(g2g3_cols)
            + len(escala_cols) + len(anyo_cols) + len(dummy_cols))

resumen = {
    'shape': list(base.shape),
    'features_ml_total': total_ml,
    'familias': {
        'indicadores_base': len(INDICADORES_BASE),
        'z_score': 1,
        'deltas_d1': len(deltas_cols),
        'crecimientos_g2g3': len(g2g3_cols),
        'escala_estructura': len(escala_cols),
        'anyo_numerico': len(anyo_cols),
        'dummies': len(dummy_cols),
    },
    'etiqueta_final_counts': base['etiqueta_final'].value_counts(
        dropna=False).to_dict(),
    'empresas_unicas': int(base['NIT_LIMPIO'].nunique()),
    'duplicados_id_anyo': int(
        base.duplicated(subset=['NIT_LIMPIO', 'ANIO']).sum()),
    'completitud_nuevos_features_pct': {
        k: float(v) for k, v in comp.round(2).to_dict().items()
    },
}
with open(ROOT / 'reports' / 'metrics' / 'features_ml_resumen.json', 'w') as f:
    json.dump(resumen, f, indent=2, ensure_ascii=False)
print('resumen ->', ROOT / 'reports' / 'metrics' / 'features_ml_resumen.json')
print()
print(json.dumps(resumen['familias'], indent=2))

resumen -> /home/Retrius/Documents/Personal Projects/trabajo final ia/reports/metrics/features_ml_resumen.json

{
  "indicadores_base": 18,
  "z_score": 1,
  "deltas_d1": 18,
  "crecimientos_g2g3": 6,
  "escala_estructura": 4,
  "anyo_numerico": 2,
  "dummies": 22
}


In [13]:
# Tabla LaTeX con los conteos por familia para el informe (Fase 10).
os.makedirs(ROOT / 'reports' / 'tables', exist_ok=True)
fam = resumen['familias']
rows = [
    (r'Indicadores financieros base', r'\textit{razon\_corriente}, ROA, $\ldots$',  fam['indicadores_base']),
    (r"Z''-Score Altman",             r'Score continuo de bancarrota',              fam['z_score']),
    (r'Variaciones interanuales',     r'$(x_t - x_{t-1}) / |x_{t-1}|$ por indicador', fam['deltas_d1']),
    (r'Crecimientos 2 y 3 anyos',     r'roa, margen\_neto, razon\_deuda',           fam['crecimientos_g2g3']),
    (r'Escala y estructura',          r'log activos, log ingresos, ratios corrientes', fam['escala_estructura']),
    (r'Temporales',                   r'anyo numerico, dummy COVID 2020',           fam['anyo_numerico']),
    (r'Dummies categoricas',          r'CIIU(12), sociedad(4), depto(6)',           fam['dummies']),
]
tex = []
tex.append(r'\begin{tabular}{lll r}')
tex.append(r'\hline')
tex.append(r'Familia & Descripcion & Numero \\')
tex.append(r'\hline')
for nombre, desc, n in rows:
    tex.append(f'{nombre} & {desc} & {n} ' + r'\\')
tex.append(r'\hline')
tex.append(r'\textbf{Total} & & \textbf{' + str(resumen['features_ml_total']) + r'} \\')
tex.append(r'\hline')
tex.append(r'\end{tabular}')
tex_str = '\n'.join(tex)
(ROOT / 'reports' / 'tables' / 'features_familias.tex').write_text(
    tex_str, encoding='utf-8',
)
print('tabla ->', ROOT / 'reports' / 'tables' / 'features_familias.tex')
print()
print(tex_str)


tabla -> /home/Retrius/Documents/Personal Projects/trabajo final ia/reports/tables/features_familias.tex

\begin{tabular}{lll r}
\hline
Familia & Descripcion & Numero \\
\hline
Indicadores financieros base & \textit{razon\_corriente}, ROA, $\ldots$ & 18 \\
Z''-Score Altman & Score continuo de bancarrota & 1 \\
Variaciones interanuales & $(x_t - x_{t-1}) / |x_{t-1}|$ por indicador & 18 \\
Crecimientos 2 y 3 anyos & roa, margen\_neto, razon\_deuda & 6 \\
Escala y estructura & log activos, log ingresos, ratios corrientes & 4 \\
Temporales & anyo numerico, dummy COVID 2020 & 2 \\
Dummies categoricas & CIIU(12), sociedad(4), depto(6) & 22 \\
\hline
\textbf{Total} & & \textbf{71} \\
\hline
\end{tabular}


## 10. Resumen para el plan

- **Shape final**: 203,104 x 77.
- **Features utilizables por ML** (sin IDs ni columnas categoricas de
  texto): 71.
- **Distribucion etiqueta_final**: 16.8% bajo / 66.4% medio / 16.8% alto.
- **Empresas unicas**: 38,245 (intactas; el filtrado de Ibague ocurre en Fase 4).
- **Completitud peor caso entre features nuevos**: ~43% (delta de
  `dias_inventario` y `rotacion_inventarios`, derivados de un denominador
  con muchos nulos en SIREM 2017). El filtrado `ANIO != 2017` propuesto
  para Fase 4 ataca el origen del problema.
- **Anomalia 2017** se conserva en el dataset: el filtrado se aplicara
  recien al hacer el split temporal (Fase 4) porque algunos diagnosticos
  de Fase 8 querran contrastar 2017 vs el resto.
- **Criterios 3.3**: los 3 pasan (no-nulos al 100% = 0; duplicados = 0;
  dummies suman 1 en CIIU(12), sociedad(4), depto(6)).